In [1]:
!pip install -q mlflow
!pip install -q --no-cache-dir --force-reinstall --no-deps git+https://github.com/RayenSansa03/aeroguard.git@main

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.

In [2]:
print("--- registre.py sur GitHub (main) ---")
!curl -s https://raw.githubusercontent.com/RayenSansa03/aeroguard/main/src/aeroguard/registre.py | grep -n "^def "
print("\n--- registre.py installé dans Colab ---")
!grep -n "^def " $(python -c "import aeroguard, os; print(os.path.dirname(aeroguard.__file__))")/registre.py

--- registre.py sur GitHub (main) ---
8:def enregistrer_version(uri_modele, nom=NOM_MODELE_ALERTE, description=None, etiquettes=None):
22:def definir_alias(alias, version, nom=NOM_MODELE_ALERTE):
29:def lire_alias(alias, nom=NOM_MODELE_ALERTE):
46:def uri_alias(alias, nom=NOM_MODELE_ALERTE):
51:def doit_promouvoir(comparaison, challenger):

--- registre.py installé dans Colab ---
8:def enregistrer_version(uri_modele, nom=NOM_MODELE_ALERTE, description=None, etiquettes=None):
22:def definir_alias(alias, version, nom=NOM_MODELE_ALERTE):
29:def lire_alias(alias, nom=NOM_MODELE_ALERTE):
46:def uri_alias(alias, nom=NOM_MODELE_ALERTE):
51:def doit_promouvoir(comparaison, challenger):


In [3]:
from google.colab import drive
drive.mount("/content/drive")

import os
import shutil
import subprocess

import mlflow
import pandas as pd
import xgboost
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from aeroguard.donnees_ml import preparer_xy

DOSSIER = "/content/drive/MyDrive/aeroguard"
DOSSIER_MLFLOW_LOCAL = "/content/mlflow"
DOSSIER_MLFLOW_DRIVE = f"{DOSSIER}/mlflow"
if os.path.exists(DOSSIER_MLFLOW_DRIVE) and not os.path.exists(DOSSIER_MLFLOW_LOCAL):
    shutil.copytree(DOSSIER_MLFLOW_DRIVE, DOSSIER_MLFLOW_LOCAL)
if os.path.exists(f"{DOSSIER}/mlruns") and not os.path.exists("/content/mlruns"):
    shutil.copytree(f"{DOSSIER}/mlruns", "/content/mlruns")
os.makedirs(DOSSIER_MLFLOW_LOCAL, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{DOSSIER_MLFLOW_LOCAL}/mlflow.db")
client = MlflowClient()

vols = pd.read_parquet(f"{DOSSIER}/data/processed/vols_ncmapss.parquet")
X_train, y_train = preparer_xy(vols, "train")
X_test, y_test = preparer_xy(vols, "test")
infos_test = vols.loc[X_test.index, ["moteur", "cycle", "RUL"]]

# Résultats officiels de la comparaison finale (leçon 4.7)
resultats_test = pd.read_csv(f"{DOSSIER}/results/v4_test_alerte.csv", index_col=0)
comparaison_test = pd.read_csv(f"{DOSSIER}/results/v4_comparaison_xgboost.csv", index_col=0)

# Version exacte du code utilisé (traçabilité)
sortie_git = subprocess.run(
    ["git", "ls-remote", "https://github.com/RayenSansa03/aeroguard.git", "refs/heads/main"],
    capture_output=True, text=True).stdout
CODE_COMMIT = sortie_git.split()[0][:7] if sortie_git else "inconnu"

print("MLflow", mlflow.__version__, "| commit du code :", CODE_COMMIT)
resultats_test[["f1_macro", "ic95_bas", "ic95_haut", "fausses_alarmes_total"]].round(3)

Mounted at /content/drive
MLflow 3.17.0 | commit du code : 6e4f9cf


,f1_macro,ic95_bas,ic95_haut,fausses_alarmes_total
xgboost,0.852,0.827,0.874,28.0
logistique,0.840,0.814,0.862,54.0
reseau_dense,0.823,0.797,0.845,51.0
cnn_1d,0.743,0.682,0.800,35.0


In [4]:
from aeroguard.registre import NOM_MODELE_ALERTE, enregistrer_version

K = 3
alerte_xgboost = xgboost.XGBClassifier()
alerte_xgboost.load_model(f"{DOSSIER}/models/xgb_binaire_v3.json")   # EXACTEMENT le modèle testé
alerte_xgboost.set_params(device="cpu")
alerte_logistique = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                  LogisticRegression(max_iter=1000)).fit(X_train, y_train)

# Ordre chronologique : la logistique (v2 du projet) puis XGBoost (v3 du projet)
CANDIDATS = {
    "logistique": {"modele": alerte_logistique, "saveur": "sklearn", "seuil": 0.50, "version_projet": "v2"},
    "xgboost":    {"modele": alerte_xgboost,    "saveur": "xgboost", "seuil": 0.81, "version_projet": "v3"},
}

versions_existantes = client.search_model_versions(f"name='{NOM_MODELE_ALERTE}'")
if versions_existantes:
    print(f"⚠️ Déjà inscrit ({len(versions_existantes)} versions) : on ne recrée rien.")
    versions = {v.tags.get("modele"): int(v.version) for v in versions_existantes}
else:
    mlflow.set_experiment("aeroguard-registre")
    versions = {}
    for nom, c in CANDIDATS.items():
        ligne = resultats_test.loc[nom]
        echantillon = X_train.head(100)
        signature = infer_signature(echantillon, c["modele"].predict_proba(echantillon)[:, 1])

        with mlflow.start_run(run_name=f"{nom}_registre"):
            mlflow.log_params({"modele": nom, "seuil": c["seuil"], "fenetre_confirmation": K,
                               "version_projet": c["version_projet"]})
            mlflow.log_metrics({"f1_macro_test": ligne["f1_macro"],
                                "ic95_bas": ligne["ic95_bas"], "ic95_haut": ligne["ic95_haut"],
                                "fausses_alarmes_test": ligne["fausses_alarmes_total"],
                                "avance_moyenne_test": ligne["avance_moyenne"]})
            mlflow.set_tags({"lecon": "D4", "jeu": "test"})
            if c["saveur"] == "xgboost":
                info = mlflow.xgboost.log_model(c["modele"], name="model", signature=signature)
            else:
                info = mlflow.sklearn.log_model(c["modele"], name="model", signature=signature,
                                                serialization_format="cloudpickle")

        versions[nom] = enregistrer_version(
            info.model_uri,
            description=(f"Alerte d'usure ({nom}) : seuil {c['seuil']}, confirmation sur {K} vols. "
                         f"F1 macro test {ligne['f1_macro']:.3f} sur 39 moteurs."),
            etiquettes={
                "modele": nom,
                "saveur": c["saveur"],
                "seuil": c["seuil"],
                "fenetre_confirmation": K,
                "version_projet": c["version_projet"],
                "f1_macro_test": round(ligne["f1_macro"], 3),
                "ic95": f"[{ligne['ic95_bas']:.3f} ; {ligne['ic95_haut']:.3f}]",
                "evalue_sur": "test, 39 moteurs (leçon 4.7)",
                "donnees": "vols_ncmapss.parquet, découpage 49/11/39",
                "code_commit": CODE_COMMIT,
            },
        )
        print(f"✅ {nom} inscrit : version {versions[nom]}")
versions

2026/10/08 23:29:54 INFO mlflow.tracking.fluent: Experiment with name 'aeroguard-registre' does not exist. Creating a new experiment.
2026/10/08 23:29:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'aeroguard-alerte'.
Created version '1' of model 'aeroguard-alerte'.


✅ logistique inscrit : version 1


Registered model 'aeroguard-alerte' already exists. Creating a new version of this model...
Created version '2' of model 'aeroguard-alerte'.


✅ xgboost inscrit : version 2


{'logistique': 1, 'xgboost': 2}

In [10]:
from aeroguard.registre import ALIAS_CHALLENGER, ALIAS_CHAMPION, definir_alias

definir_alias(ALIAS_CHAMPION, versions["xgboost"])
definir_alias(ALIAS_CHALLENGER, versions["logistique"])


def afficher_catalogue():
    """Toutes les versions du modèle, avec leurs alias (relus version par version)."""
    lignes = []
    for v in client.search_model_versions(f"name='{NOM_MODELE_ALERTE}'"):
        complete = client.get_model_version(NOM_MODELE_ALERTE, v.version)  # contient les alias
        lignes.append({
            "version": int(complete.version),
            "modele": complete.tags.get("modele"),
            "alias": ", ".join(complete.aliases),
            "seuil": complete.tags.get("seuil"),
            "f1_macro_test": complete.tags.get("f1_macro_test"),
            "ic95": complete.tags.get("ic95"),
            "code_commit": complete.tags.get("code_commit"),
        })
    return pd.DataFrame(lignes).sort_values("version")


afficher_catalogue()

,version,modele,alias,seuil,f1_macro_test,ic95,code_commit
1,1,logistique,challenger,0.5,0.84,[0.814 ; 0.862],6e4f9cf
0,2,xgboost,champion,0.81,0.852,[0.827 ; 0.874],6e4f9cf


In [6]:
from aeroguard.metier import evaluer_alerte
from aeroguard.registre import lire_alias, uri_alias

CHARGEURS = {"xgboost": mlflow.xgboost.load_model, "sklearn": mlflow.sklearn.load_model}


def charger_par_alias(alias):
    """Charge le modèle qui porte l'alias, avec sa notice (seuil, k, F1 attendu...)."""
    infos = lire_alias(alias)
    if infos is None:
        raise ValueError(f"Aucune version ne porte l'alias « {alias} ».")
    modele = CHARGEURS[infos["etiquettes"]["saveur"]](uri_alias(alias))
    return modele, infos


champion, infos_champion = charger_par_alias(ALIAS_CHAMPION)
notice = infos_champion["etiquettes"]
print(f"Champion : version {infos_champion['version']} ({notice['modele']}), "
      f"seuil {notice['seuil']}, k = {notice['fenetre_confirmation']}")

probas = champion.predict_proba(X_test)[:, 1]
scores = evaluer_alerte(y_test, probas, float(notice["seuil"]), infos_test,
                        k=int(notice["fenetre_confirmation"]), col_moteur="moteur")
attendu = float(notice["f1_macro_test"])
print(f"F1 macro recalculé : {scores['f1_macro']:.3f} | attendu : {attendu:.3f}")
assert abs(scores["f1_macro"] - attendu) < 1e-3, "Le registre ne contient pas le modèle testé !"
print("✅ Test de fumée réussi : le registre contient bien le modèle évalué en 4.7")

Champion : version 2 (xgboost), seuil 0.81, k = 3
F1 macro recalculé : 0.852 | attendu : 0.852
✅ Test de fumée réussi : le registre contient bien le modèle évalué en 4.7


In [7]:
from aeroguard.registre import doit_promouvoir

decisions = pd.DataFrame({
    "ecart_moyen": comparaison_test["ecart_moyen"],
    "intervalle_ecart": [f"[{b:+.3f} ; {h:+.3f}]" for b, h in
                         zip(comparaison_test["ecart_bas"], comparaison_test["ecart_haut"])],
    "proba_meilleur": comparaison_test["proba_meilleur"],
    "promouvoir": [doit_promouvoir(comparaison_test, nom) for nom in comparaison_test.index],
})
print("Champion actuel : xgboost — un challenger doit le battre de façon PROUVÉE.")
decisions.round(3)

Champion actuel : xgboost — un challenger doit le battre de façon PROUVÉE.


,ecart_moyen,intervalle_ecart,proba_meilleur,promouvoir
modele,,,,
logistique,-0.013,[-0.038 ; +0.011],0.15,False
xgboost,0.000,[+0.000 ; +0.000],0.00,False
reseau_dense,-0.029,[-0.042 ; -0.015],0.00,False
cnn_1d,-0.108,[-0.164 ; -0.055],0.00,False


In [8]:
dossiers_artifacts = set()
for experience in mlflow.search_experiments():
    emplacement = experience.artifact_location.replace("file://", "")
    if emplacement.startswith("/content") and not emplacement.startswith("/content/drive"):
        dossiers_artifacts.add(os.path.dirname(emplacement))

shutil.copytree(DOSSIER_MLFLOW_LOCAL, DOSSIER_MLFLOW_DRIVE, dirs_exist_ok=True)
for dossier in dossiers_artifacts:
    if os.path.exists(dossier):
        shutil.copytree(dossier, f"{DOSSIER}/{os.path.basename(dossier)}", dirs_exist_ok=True)
print("✅ MLflow (runs + registre) sauvegardé sur Drive")

✅ MLflow (runs + registre) sauvegardé sur Drive


In [9]:
import time
import urllib.request

from google.colab import output

PORT_INTERFACE = 5000
FICHIER_JOURNAL = "/content/mlflow_serveur.log"
subprocess.run(["pkill", "-f", "mlflow server"], check=False)
time.sleep(2)

commande = ["mlflow", "server", "--backend-store-uri", mlflow.get_tracking_uri(),
            "--host", "127.0.0.1", "--port", str(PORT_INTERFACE)]
aide = subprocess.run(["mlflow", "server", "--help"], capture_output=True, text=True).stdout
if "--allowed-hosts" in aide:
    commande += ["--allowed-hosts", "*"]
if "--cors-allowed-origins" in aide:
    commande += ["--cors-allowed-origins", "*"]
if "--x-frame-options" in aide:
    commande += ["--x-frame-options", "NONE"]

journal = open(FICHIER_JOURNAL, "w")
serveur_mlflow = subprocess.Popen(commande, stdout=journal, stderr=subprocess.STDOUT)

serveur_pret = False
for seconde in range(60):
    if serveur_mlflow.poll() is not None:
        break
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT_INTERFACE}", timeout=2)
        serveur_pret = True
        break
    except Exception:
        time.sleep(1)

if serveur_pret:
    print(f"✅ Serveur prêt après {seconde} s")
    output.serve_kernel_port_as_iframe(PORT_INTERFACE, height=800)
else:
    journal.flush()
    print("❌ Le serveur ne répond pas. Journal :")
    print(open(FICHIER_JOURNAL).read()[-3000:])

✅ Serveur prêt après 32 s


<IPython.core.display.Javascript object>